In [1]:
from IPython.display import display, HTML
display(HTML("""
<style>
div.container{width:85% !important;}
div.cell.code_cell.rendered{width:100%;}
div.input_prompt{padding:0px;}
div.CodeMirror {font-family:Consolas; font-size:20pt;}
div.output {font-size:12pt; font-weight:bold;}
div.input {font-family:Consolas; font-size:20pt;}
div.prompt {min-width:70px;}
div#toc-wrapper{padding-top:120px;}
div.text_cell_render ul li{font-size:12pt;padding:5px;}
table.dataframe{font-size:22px;}

</style>
"""))

# 백터DB:Chroma VS Pinecone
- Chroma : 인메모리 vector DB, 로컬 vector DB, collection단위로 저장
- Pinecone : 클라우드 vector DB, index단위로 저장
(http://www.pinecone.io)에서 api key받아 .env추가(PINECONE_API_KEY등록)

# 0. 패키지 설치

In [2]:
%pip install -q pinecone langchain-pinecone #--no-warn-script-location

Note: you may need to restart the kernel to use updated packages.


  Consider adding this directory to PATH or, if you prefer to suppress this warning, use --no-warn-script-location.
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
transformers 4.57.3 requires tokenizers<=0.23.0,>=0.22.0, but you have tokenizers 0.20.3 which is incompatible.


In [4]:
%pip show langchain-pinecone

Name: langchain-pinecone
Version: 0.2.13
Summary: An integration package connecting Pinecone and LangChain
Home-page: 
Author: 
Author-email: 
License: MIT
Location: c:\users\admin\anaconda3\envs\llm\lib\site-packages
Requires: httpx, langchain-core, langchain-openai, numpy, pinecone, pydantic, simsimd
Required-by: 
Note: you may need to restart the kernel to use updated packages.


# 1. knowlege base 구성을 위한 데이터 생성

In [7]:

from langchain_community.document_loaders import Docx2txtLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
loader = Docx2txtLoader("./tex_docx/소득세법(법률)(제21221호)(20260701).docx")
text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=1500, # 문서를 쪼갤때 1500글자씩 chunking
    chunk_overlap=200, # 200문자씩 overlap 하여 chunking
    # separators=["\n\n","\n"," ",""]
)
# Recursive(재귀적으로) 다음의 순서대로 chunking 시도 : "\n\n","\n"," ",""
documents = loader.load_and_split(text_splitter=text_splitter)
len(documents)

195

In [15]:
from dotenv import load_dotenv
from langchain_upstage import UpstageEmbeddings
load_dotenv()
embedding = UpstageEmbeddings(
    #api_key = os.getenv('OPENAI_API_KEY'),
    model = 'solar-embedding-1-large-passage'
)

In [ ]:
from dotenv import load_dotenv
from langchain_openai import OpenAIEmbeddings
load_dotenv()
embeding = OpenAIEmbeddings(
    #api_key = os.getenv('OPENAI_API_KEY'),
    model = 'text-embedding-3-large'
)

In [16]:
len(embedding.embed_query("샘플"))

4096

In [ ]:
# 모델마다 코사인유사도를 쓰는지 document
# https://console.upstage.ai/docs/capabilities/embed
# https://developers.openai.com/api/docs/guides/embeddings

In [19]:
%%time
# pinecone vector database
from pinecone import Pinecone
from langchain_pinecone import PineconeVectorStore
import os
pc = Pinecone(
    #api_key=os.getenv(PINECONE_API_KEY)
)
# 데이터를 처음 저장할때
# index_name = 'tax-index'
# database = PineconeVectorStore.from_documents(
#     documents=documents,
#     embedding=embedding,
#     index_name=index_name
# )

# 업로드된 백터DB를 load할때
database = PineconeVectorStore(
    embedding=embedding,
    index_name=index_name
)

CPU times: total: 0 ns
Wall time: 1.07 ms


In [21]:
%pip install -q ipywidgets

Note: you may need to restart the kernel to use updated packages.


In [22]:
pip show ipywidgets

Name: ipywidgets
Version: 8.1.9
Summary: Jupyter interactive widgets
Home-page: http://jupyter.org
Author: Jupyter Development Team
Author-email: jupyter@googlegroups.com
License: BSD 3-Clause License
Location: c:\users\admin\anaconda3\envs\llm\lib\site-packages
Requires: comm, ipython, jupyterlab_widgets, traitlets, widgetsnbextension
Required-by: 
Note: you may need to restart the kernel to use updated packages.


# 2. 답변생성을 위한 Retrieval

In [28]:
query = "연봉 5천만원인 직장인은 소득세를 얼마낼까요?"
retrieved_docs = database.similarity_search(query,k=3)


In [30]:
retrieved_doc="\n--\n".join([doc.page_content.replace('\n\n',' ') for doc in retrieved_docs])
# print(retrieved_doc)

# 3.

In [20]:
from langchain_openai import Chatupstage
llm = Chatupstage(model='gpt-4.1-nano')

In [ ]:
from langchain_upstage import ChatOpenAI
llm = ChatOpenAI(model='solar-pro2')

In [31]:
prompt = f"""[identity]
- 당신은 최고의 한국 소득세법 전문가이다. [context]를 참고해서 사용자 질문에 답하세요.[context]는 다음과 같아요
{retrieved_doc}
- 질문 :{query}"""

In [32]:
ai_message = llm.invoke(prompt)
print(ai_message.content)

연봉 5천만원인 직장인의 소득세 산출을 위해 기본적인 계산 과정을 설명드리겠습니다. 단, 정확한 세액은 공제 및 세액공제 대상 여부에 따라 차이가 있을 수 있으니 참고용으로 봐주시기 바랍니다.

【기본 가정】
- 근로소득 연봉: 50,000,000원
- 국민연금, 건강보험, 고용보험 등 4대 보험료 공제는 공제액이 별도로 정해져 있으며, 일반적으로 총 급여의 약 8-10% 수준입니다. 
- 표준공제 및 기본공제만 적용한 계산입니다.
- 자녀 세액공제, 보험료 공제, 특별세액공제 등 기타 공제는 고려하지 않습니다.

1. 총급여액 계산
- 연봉: 50,000,000원

2. 근로소득공제
- 제47조에 따르면, 근로소득공제액은 2,000만원 초과시 2,000만원 한도로 적용.
- 공제액: 2,000만원

3. 과세표준 산출
- 총급여액: 50,000,000원
- 근로소득공제: -20,000,000원
- 과세표준: 30,000,000원

4. 기본세율 적용
- 1,200만원 이하: 6%
- 1,200만원 초과 ~ 4,600만원 이하: 15%
- 4,600만원 초과 ~ 8,800만원 이하: 24%
- 8,800만원 초과 ~ 1억 5천만원 이하: 35%
- 1억 5천만원 초과: 38%

이 경우 과세표준 30,000,000원은 1,200만원 + 1,800만원이 초과하는 구간입니다.

세율 계산은 구간별로 적용하므로,

- 1,200만원까지: 1,200만원 × 6% = 72만원
- 나머지 (30,000,000 - 12,000,000) = 18,000,000원
  - 15% 세율 적용 구간: 15% × 18,000,000원 = 2,700,000원

하지만, 과세소득 세율은 누진세 구조이기 때문에 정확한 세액은 누진세 계산표를 바탕으로 산출됩니다. 

대략적인 계산:
- 세액 = (과세표준 구간별 누진세 액수의 합)

대략 산출
- 1,200만원까지: 72만원
- 나머지 1,800만원 구간: (1,800만원) × 15% = 270만원

하지만 이 계산은 단순화를 위해서입니다